In [1]:
import os
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import cv2

os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

2026-06-19 10:15:43.831598: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-19 10:15:43.858876: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-19 10:15:43.858912: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-19 10:15:43.859878: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-19 10:15:43.865012: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-19 10:15:43.865981: I tensorflow/core/platform/cpu_feature_guard.cc:1

Segmentation Models: using `tf.keras` framework.


In [2]:
metadata = pd.read_csv("data/metadata.csv")

metadata.head()

,Image,Mask
0,0.jpg,0.png
1,1.jpg,1.png
2,2.jpg,2.png
3,3.jpg,3.png
4,4.jpg,4.png


In [3]:
IMAGE_DIR = "data/Image"
MASK_DIR = "data/Mask"

image_paths = [
    os.path.join(IMAGE_DIR, img_name)
    for img_name in metadata["Image"]
]

mask_paths = [
    os.path.join(MASK_DIR, mask_name)
    for mask_name in metadata["Mask"]
]

In [4]:
from sklearn.model_selection import train_test_split

train_images, val_images, train_masks, val_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_images))
print("Validation samples:", len(val_images))

Training samples: 232
Validation samples: 58


In [5]:
print(train_images[:5])
print(train_masks[:5])

['data/Image/6.jpg', 'data/Image/1007.jpg', 'data/Image/1021.jpg', 'data/Image/1059.jpg', 'data/Image/2024.jpg']
['data/Mask/6.png', 'data/Mask/1007.png', 'data/Mask/1021.png', 'data/Mask/1059.png', 'data/Mask/2024.png']


In [6]:
def load_single(img_path, mask_path, augment=False):
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (256, 256))

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (256, 256))
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    if augment:
        if np.random.rand() > 0.5:
            image = np.fliplr(image)
            mask  = np.fliplr(mask)
        if np.random.rand() > 0.5:
            image = np.flipud(image)
            mask  = np.flipud(mask)
        factor = np.random.uniform(0.7, 1.3)
        image  = np.clip(image * factor, 0, 255).astype(np.uint8)

    image = image.astype(np.float32) / 255.0
    return image, mask

all_images, all_masks = [], []
for img_p, msk_p in zip(train_images, train_masks):
    img, msk = load_single(img_p, msk_p, augment=False)
    all_images.append(img)
    all_masks.append(msk)
    for _ in range(3):
        img_aug, msk_aug = load_single(img_p, msk_p, augment=True)
        all_images.append(img_aug)
        all_masks.append(msk_aug)

# val — no augmentation
val_images_arr, val_masks_arr = [], []
for img_p, msk_p in zip(val_images, val_masks):
    img, msk = load_single(img_p, msk_p, augment=False)
    val_images_arr.append(img)
    val_masks_arr.append(msk)

X_train = np.array(all_images)
y_train = np.array(all_masks)
X_val   = np.array(val_images_arr)
y_val   = np.array(val_masks_arr)

print(f"Train samples: {len(X_train)}")  # expect ~928
print(f"Val samples:   {len(X_val)}")    # expect 58

Train samples: 928
Val samples:   58


In [7]:
BATCH_SIZE = 8

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(200).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# sanity check
for images, masks in train_dataset.take(1):
    print(f"Image batch: {images.shape}")  # (8, 256, 256, 3)
    print(f"Mask batch:  {masks.shape}")   # (8, 256, 256, 1)

Image batch: (8, 256, 256, 3)
Mask batch:  (8, 256, 256, 1)


In [8]:
# split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42
)

In [9]:
# tf datasets — no map, no numpy_function, no AutoGraph issues
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


KeyboardInterrupt: 

In [ ]:
# sanity check
for images, masks in train_dataset.take(1):
    print(f"Image batch: {images.shape}")  # (8, 256, 256, 3)
    print(f"Mask batch:  {masks.shape}")   # (8, 256, 256, 1)
    print(f"Mask unique: {np.unique(masks.numpy())}")  # [0. 1.]

Image batch: (8, 256, 256, 3)
Mask batch:  (8, 256, 256, 1)
Mask unique: [0. 1.]


In [ ]:
for images, masks in train_dataset.take(1):
    print(images.shape)
    print(masks.shape)

(8, 256, 256, 3)
(8, 256, 256, 1)


In [ ]:
model = sm.Unet(
    'resnet34',
    encoder_weights='imagenet',
    classes=1,
    activation='sigmoid'
)

In [ ]:
from keras.optimizers import Adam

model.compile(
    optimizer = Adam(learning_rate = 1e-4),
    loss= sm.losses.bce_dice_loss,
    metrics=[
        sm.metrics.iou_score,
        sm.metrics.f1_score
    ]
)

In [ ]:
callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_f1-score",
        mode="max",
        save_best_only=True
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_f1-score",
        mode="max",
        patience=8,
        restore_best_weights=True
    )

]

for images, masks in train_dataset.take(1):
    print(images.shape)
    print(masks.shape)

pred = model.predict(images)
print(pred.shape)

(8, 256, 256, 3)
(8, 256, 256, 1)
1/1 [==============================] - 1s 735ms/step
(8, 256, 256, 1)


In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=callbacks
)

BATCH_SIZE = 8

Epoch 1/30
29/29 [==============================] - 79s 3s/step - loss: 1.0246 - iou_score: 0.4030 - f1-score: 0.5721 - val_loss: 1.2519 - val_iou_score: 0.3163 - val_f1-score: 0.4795
Epoch 2/30
29/29 [==============================] - 67s 2s/step - loss: 0.7141 - iou_score: 0.5245 - f1-score: 0.6867 - val_loss: 1.3141 - val_iou_score: 0.3614 - val_f1-score: 0.5296
Epoch 3/30
29/29 [==============================] - 64s 2s/step - loss: 0.5949 - iou_score: 0.5820 - f1-score: 0.7343 - val_loss: 1.4628 - val_iou_score: 0.3848 - val_f1-score: 0.5542
Epoch 4/30
29/29 [==============================] - 64s 2s/step - loss: 0.5112 - iou_score: 0.6303 - f1-score: 0.7713 - val_loss: 1.9570 - val_iou_score: 0.4028 - val_f1-score: 0.5724
Epoch 5/30
29/29 [==============================] - 64s 2s/step - loss: 0.4629 - iou_score: 0.6582 - f1-score: 0.7928 - val_loss: 2.2475 - val_iou_score: 0.4092 - val_f1-score: 0.5786
Epoch 6/30
29/29 [==============================] - 63s 2s/step - loss: 0.4334 -

KeyboardInterrupt: 